# Experiment: Public Case Data Release Gate

Objective: model a small release gate for case-linked data before it enters the public repo.

Success criteria:
- raw clinical records and local paths are blocked;
- direct identifiers are blocked;
- patient-specific treatment action claims are blocked;
- de-identified summaries can pass only with uncertainty and clinician-review boundaries.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Final

RELEASE_ORDER: Final = (
    "private_raw_record",
    "public_blocked_identifier",
    "public_blocked_treatment_claim",
    "deidentified_extract_candidate",
    "source_link_only",
    "public_summary_ready",
)

RELEASE_ORDER

## Gate Model

This model is not a legal de-identification engine. It is a reproducible repo gate for obvious public-release mistakes.


In [ ]:
@dataclass(frozen=True)
class ReleaseSignal:
    """A single privacy or claim signal found in a candidate public artifact."""

    name: str
    release_class: str
    present: bool
    reason: str


def release_decision(signals: list[ReleaseSignal]) -> dict[str, object]:
    """Return the strongest public-release class for a candidate artifact."""
    present_signals = [signal for signal in signals if signal.present]

    if not present_signals:
        return {
            "decision": "public_summary_ready",
            "blockers": [],
            "review_notes": [],
        }

    ordered = sorted(
        present_signals,
        key=lambda signal: RELEASE_ORDER.index(signal.release_class),
    )
    decision = ordered[0].release_class
    blockers = [
        signal.name
        for signal in ordered
        if signal.release_class.startswith("public_blocked")
        or signal.release_class == "private_raw_record"
    ]
    review_notes = [signal.reason for signal in ordered]

    return {
        "decision": decision,
        "blockers": blockers,
        "review_notes": review_notes,
    }

## Scenario Smoke Test

The scenarios mirror the mistakes this repo most needs to prevent: raw files, identifiers, and therapy-action wording.


In [ ]:
scenarios = {
    "raw_pdf_with_local_path": [
        ReleaseSignal(
            name="raw_pdf",
            release_class="private_raw_record",
            present=True,
            reason="Raw medical PDFs stay outside Git.",
        ),
        ReleaseSignal(
            name="local_source_path",
            release_class="private_raw_record",
            present=True,
            reason="Local source paths are private working context.",
        ),
    ],
    "hospital_identifier": [
        ReleaseSignal(
            name="hospital_record_number",
            release_class="public_blocked_identifier",
            present=True,
            reason="Direct record identifiers cannot be public.",
        )
    ],
    "patient_specific_action_claim": [
        ReleaseSignal(
            name="therapy_action",
            release_class="public_blocked_treatment_claim",
            present=True,
            reason="Treatment actions require clinician review outside this repo.",
        )
    ],
    "deidentified_question_packet": [],
}

scenario_results = {
    name: release_decision(signals) for name, signals in scenarios.items()
}

assert scenario_results["raw_pdf_with_local_path"]["decision"] == "private_raw_record"
assert (
    scenario_results["hospital_identifier"]["decision"] == "public_blocked_identifier"
)
assert (
    scenario_results["patient_specific_action_claim"]["decision"]
    == "public_blocked_treatment_claim"
)
assert (
    scenario_results["deidentified_question_packet"]["decision"]
    == "public_summary_ready"
)

scenario_results

## Interpretation

The strongest blockers are intentionally simple. If an artifact contains a raw medical record, a direct identifier, a local file path, or a patient-specific treatment action, it should not be committed. De-identified question packets can be public only when they preserve uncertainty labels and clinician-review boundaries.

Durable repo outputs:
- `templates/public-case-data-release-checklist.md`
- `research/thalassemia/findings/2026-04-28-public-case-data-release-gate.md`
- `scripts/check_public_repo.py`
